# MavenMagnet – Data Scientist technical exercise
## Tagging posts with Level 1 / Level 2 factors using an LLM

**Goal:** `Process.xlsx` has 190 posts with empty `Level 1` / `Level 2` columns. Using the labelled `Training.xlsx` as a guide, I need to fill them in (one row per mapping, same format as Training). BERT is not allowed and I have to use an LLM.

**My approach (short version)**
1. Look at the training data and pull out the valid Level 1 → Level 2 taxonomy.
2. Use a Groq-hosted LLM with **few-shot prompting**: for every post I fetch the most similar labelled training posts (TF-IDF cosine similarity) and show them to the LLM as examples, together with the full taxonomy.
3. Force JSON output and **throw away any label pair that is not in the taxonomy** (stops hallucinated labels).
4. Hold out 25% of the training posts as a validation set and measure precision / recall / F1, so the accuracy number is real and not a guess.
5. Compare two LLMs and a simple TF-IDF + Logistic Regression baseline, pick the best, and run it on `Process.xlsx`.

Put `Training.xlsx` and `Process.xlsx` in the same folder as this notebook and you need a free Groq API key from https://console.groq.com.

In [1]:
# Only needed the first time
!pip install -q groq pandas scikit-learn openpyxl

In [2]:
import os, re, json, time, random, getpass
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from groq import Groq

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

## 1. Config

In [3]:
DATA_DIR = "."                       # folder with the two excel files
TRAIN_FILE = os.path.join(DATA_DIR, "Training.xlsx")
PROCESS_FILE = os.path.join(DATA_DIR, "Process.xlsx")
OUTPUT_FILE = "Process_labelled.xlsx"
OUTPUT_CSV = "Process_labelled.csv"

# Models I want to use (both are on Groq's free tier).
# The next cell checks them against my key and swaps in an available one if needed.
BIG_MODEL = "llama-3.3-70b-versatile"
SMALL_MODEL = "llama-3.1-8b-instant"
COMPARE_MODELS = False   # True = run both models on the validation set (slower, uses more free tokens)

K_EXAMPLES = 3         # similar training posts shown to the LLM per post
BATCH_SIZE = 10        # posts per request (the taxonomy is sent once per request, so this saves tokens)
VAL_FRACTION = 0.25    # share of training posts kept aside for validation
VAL_LIMIT = 40         # max validation posts sent to the LLM (keeps the free-tier usage low)
SEED = 42
TPM_LIMIT = 10000      # tokens per minute I allow myself (free tier is about 12k)
MAX_WAIT = 30          # if Groq asks for a longer wait than this, stop waiting and use the fallback
CACHE_FILE = "llm_cache_v2.json"  # every LLM answer is saved here, so a crash does not cost me the work

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your Groq API key: ")
client = Groq()

Paste your Groq API key:  ········


In [4]:
# Check which models my key can really use, so a wrong model name fails here and not after hours of retries
available = sorted(m.id for m in client.models.list().data)
print("models available to my key:", available)

chat_models = [m for m in available if not any(x in m for x in ["whisper", "guard", "tts", "orpheus"])]
if not chat_models:
    raise SystemExit("No chat model available for this key. Check the Groq account.")

def pick(preferred, backups):
    for m in [preferred] + backups:
        if m in available:
            return m
    return chat_models[0]

BIG_MODEL = pick(BIG_MODEL, ["llama-3.1-8b-instant", "meta-llama/llama-4-scout-17b-16e-instruct", "openai/gpt-oss-20b"])
SMALL_MODEL = pick(SMALL_MODEL, [BIG_MODEL])
MODELS_TO_RUN = [SMALL_MODEL, BIG_MODEL] if (COMPARE_MODELS and SMALL_MODEL != BIG_MODEL) else [BIG_MODEL]
print("models in use:", MODELS_TO_RUN)

models available to my key: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']
models in use: ['openai/gpt-oss-20b']


## 2. Load and understand the data

In [5]:
train_raw = pd.read_excel(TRAIN_FILE)
process_raw = pd.read_excel(PROCESS_FILE)

print("Training :", train_raw.shape, "| Process:", process_raw.shape)
print("Training columns:", train_raw.columns.tolist())
print("Process columns :", process_raw.columns.tolist())
train_raw.head(8)

Training : (512, 3) | Process: (190, 3)
Training columns: ['Post', 'Level 1', 'Level 2']
Process columns : ['Post', 'Level 1', 'Level 2']


,Post,Level 1,Level 2
0,a bit pricey product but its worth it,Pricing,Value for money
1,A drop or two are enough to moisture your entire face.,Moisturization,Intensity
2,A drop or two are enough to moisture your entire face.,Pricing,Value for money
3,A liitle bit goes a long way and it soothes skin,Feel,Soothening
4,A liitle bit goes a long way and it soothes skin,Pricing,Value for money
5,A wonderfully light cleaner which is making my face glow and stay free of clogged pores.,Acne,Minimize Pores
6,A wonderfully light cleaner which is making my face glow and stay free of clogged pores.,Finish,Light Weight
7,A wonderfully light cleaner which is making my face glow and stay free of clogged pores.,Tone,Glow


In [6]:
# basic cleaning: drop empty rows, strip spaces, remove exact duplicate mappings
train = train_raw.dropna(subset=["Post", "Level 1", "Level 2"]).copy()
for col in ["Post", "Level 1", "Level 2"]:
    train[col] = train[col].astype(str).str.strip()
train = train.drop_duplicates().reset_index(drop=True)

process_posts = process_raw["Post"].dropna().astype(str).str.strip().drop_duplicates().tolist()

print("Training rows after cleaning:", len(train))
print("Unique training posts       :", train["Post"].nunique())
print("Unique posts to label       :", len(process_posts))
print("Level 1 factors:", train["Level 1"].nunique(), "| Level 2 factors:", train["Level 2"].nunique())

Training rows after cleaning: 510
Unique training posts       : 266
Unique posts to label       : 190
Level 1 factors: 17 | Level 2 factors: 42


In [7]:
labels_per_post = train.groupby("Post").size()
print("Labels per post (training):")
print(labels_per_post.describe().round(2))
print()
print("Level 1 frequency:")
print(train["Level 1"].value_counts())

Labels per post (training):
count    266.00
mean       1.92
std        0.72
min        1.00
25%        1.00
50%        2.00
75%        2.00
max        4.00
dtype: float64

Level 1 frequency:
Level 1
Tone              90
Efficacy          83
Sensory           57
Texture           48
Pricing           39
Finish            37
Safety            36
Moisturization    35
Feel              32
Acne              20
Cleansing         10
Anti-Aging         9
Relief             7
Lasting            4
Ingredients        1
Health             1
Practicality       1
Name: count, dtype: int64


Takeaways: the data is small (a few hundred posts), a post usually has 1–3 mappings, and some factors are very rare (some Level 1 values appear only once or twice). A classic classifier will struggle with the rare ones, while an LLM that is told the taxonomy can still use them. That is the main reason I am going with an LLM.

In [8]:
# Taxonomy: Level 1 -> list of Level 2 children (taken straight from the training file)
taxonomy = train.groupby("Level 1")["Level 2"].apply(lambda s: sorted(set(s))).to_dict()
valid_pairs = {(l1, l2) for l1, kids in taxonomy.items() for l2 in kids}
canon = {(a.lower(), b.lower()): (a, b) for a, b in valid_pairs}   # for case-insensitive matching

for l1, kids in taxonomy.items():
    print(f"{l1:15s} -> {', '.join(kids)}")
print("\nTotal valid (Level 1, Level 2) pairs:", len(valid_pairs))

# gold labels per post  ->  {post: {(l1, l2), ...}}
gold = {}
for post, l1, l2 in train[["Post", "Level 1", "Level 2"]].itertuples(index=False):
    gold.setdefault(post, set()).add((l1, l2))

Acne            -> Acne Control, Minimize Pores
Anti-Aging      -> Dullness, Elasticity, Fine Lines
Cleansing       -> Dirt Removal, Exfoliation, Oil Removal
Efficacy        -> Prolonged Use, Quick Results, Visible Difference
Feel            -> Cooling Sensation, Fresh, Soothening
Finish          -> Absorption, Clean, Dewy, Light Weight
Health          -> Texture
Ingredients     -> Vitamin C
Lasting         -> Long Stay Period
Moisturization  -> Hydration, Intensity, Nourishment
Practicality    -> Availability
Pricing         -> Affordability, Discounts, Value for money
Relief          -> Damaged Skin, Redness
Safety          -> Mildness, Side Effects
Sensory         -> Format, Smell, Touch
Texture         -> Dryness Control, Elasticity, Oil Balance, Smoothness, Softness
Tone            -> Depigmentation, Glow, Skin Lightening

Total valid (Level 1, Level 2) pairs: 43


## 3. Train / validation split
I split by **unique post** (not by row) so that the same post never sits on both sides, which would leak labels.

In [9]:
all_posts = sorted(gold)
rng = random.Random(SEED)
rng.shuffle(all_posts)

n_val = int(len(all_posts) * VAL_FRACTION)
val_posts = all_posts[:n_val]
fit_posts = all_posts[n_val:]
val_posts = val_posts[:VAL_LIMIT]     # only this many go to the LLM, the rest of the held-out posts are not used
print(f"fit posts: {len(fit_posts)} | validation posts: {len(val_posts)}")

fit posts: 200 | validation posts: 40


## 4. Finding similar labelled examples (for few-shot prompts)
For each new post I look up the `K` most similar labelled posts using TF-IDF cosine similarity and show them to the LLM. This is just a search step, **the labelling itself is done by the LLM**. Showing the LLM how similar posts were tagged also teaches it the quirks of this particular labelling style.

In [10]:
class ExampleRetriever:
    def __init__(self, posts, gold):
        self.posts = list(posts)
        self.gold = gold
        self.vec = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", sublinear_tf=True)
        self.mat = self.vec.fit_transform(self.posts)

    def nearest(self, text, k=10):
        sims = cosine_similarity(self.vec.transform([text]), self.mat).ravel()
        top = np.argsort(-sims)[:k]
        return [self.posts[i] for i in top]

retr_fit = ExampleRetriever(fit_posts, gold)      # only uses the 75% fit posts -> honest validation
retr_full = ExampleRetriever(all_posts, gold)     # uses everything -> used for the real Process file

demo = val_posts[0]
print("Post:", demo)
print("Gold:", sorted(gold[demo]))
print("\nMost similar training posts:")
for p in retr_fit.nearest(demo, 3):
    print(" -", p, "->", sorted(gold[p]))

Post: I see my skin firmer and more healthy, the relaxing fragrance is a plus especially for a nighttime routine.
Gold: [('Anti-Aging', 'Elasticity'), ('Sensory', 'Smell'), ('Tone', 'Glow')]

Most similar training posts:
 - but found it really relaxing and what a cooling effect! -> [('Feel', 'Cooling Sensation'), ('Feel', 'Soothening')]
 - skin looks very plump, hydrated, glowing and healthy. -> [('Moisturization', 'Hydration'), ('Texture', 'Elasticity'), ('Tone', 'Glow')]
 - Amazing!!! Very light and good fragrance ( might not be good for sensitive skin) -> [('Finish', 'Light Weight'), ('Safety', 'Mildness'), ('Sensory', 'Smell')]


## 5. Prompt, LLM call and output validation

In [11]:
taxonomy_text = "\n".join(f"- {l1}: {', '.join(kids)}" for l1, kids in taxonomy.items())

SYSTEM_PROMPT = f"""You are an expert annotator for consumer reviews of skincare products.
Each short review text ("post") is tagged with one or more factors. A factor has a Level 1 (broad topic)
and a Level 2 (more specific, nested under that Level 1).

You may ONLY use Level 1 / Level 2 combinations from this taxonomy (format - Level 1: allowed Level 2 values):
{taxonomy_text}

Rules:
- Tag only what the post actually talks about. Do not guess or over-tag. Most posts have 1 to 3 labels, never more than 4.
- Level 2 must belong to the Level 1 it is paired with. The same Level 2 name can exist under different Level 1 values, so pair them correctly.
- Follow the style of the labelled examples you are given.
- You get several posts at once. Reply with JSON only. Keys are the post numbers as strings, values are lists of labels:
  {{"1": [{{"level1": "<Level 1>", "level2": "<Level 2>"}}], "2": [{{"level1": "<Level 1>", "level2": "<Level 2>"}}]}}
"""

def fmt_labels(pairs):
    return json.dumps([{"level1": a, "level2": b} for a, b in sorted(pairs)])

def build_batch_prompt(batch, retriever, k):
    parts = []
    for i, post in enumerate(batch, 1):
        parts.append(f"Post {i}: {post}")
        parts.append("Similar labelled posts:")
        for ex in retriever.nearest(post, k)[::-1]:       # most similar example goes last
            parts.append(f'- "{ex[:150]}" => {fmt_labels(retriever.gold[ex])}')
        parts.append("")
    return "Tag each post below.\n\n" + "\n".join(parts)

print(SYSTEM_PROMPT)
print("-" * 80)
print(build_batch_prompt([demo], retr_fit, 3))

You are an expert annotator for consumer reviews of skincare products.
Each short review text ("post") is tagged with one or more factors. A factor has a Level 1 (broad topic)
and a Level 2 (more specific, nested under that Level 1).

You may ONLY use Level 1 / Level 2 combinations from this taxonomy (format - Level 1: allowed Level 2 values):
- Acne: Acne Control, Minimize Pores
- Anti-Aging: Dullness, Elasticity, Fine Lines
- Cleansing: Dirt Removal, Exfoliation, Oil Removal
- Efficacy: Prolonged Use, Quick Results, Visible Difference
- Feel: Cooling Sensation, Fresh, Soothening
- Finish: Absorption, Clean, Dewy, Light Weight
- Health: Texture
- Ingredients: Vitamin C
- Lasting: Long Stay Period
- Moisturization: Hydration, Intensity, Nourishment
- Practicality: Availability
- Pricing: Affordability, Discounts, Value for money
- Relief: Damaged Skin, Redness
- Safety: Mildness, Side Effects
- Sensory: Format, Smell, Touch
- Texture: Dryness Control, Elasticity, Oil Balance, Smoothnes

In [12]:
state = {"pause": 0, "stopped": set()}     # stopped = models where I gave up calling the API

def ask_llm(model, system, user, retries=3):
    if model in state["stopped"]:
        return ""
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                temperature=0,
                max_tokens=1200,
                response_format={"type": "json_object"},
            )
            used = resp.usage.total_tokens if resp.usage else 4000
            state["pause"] = used / TPM_LIMIT * 60       # wait only as long as the token budget needs
            return resp.choices[0].message.content
        except Exception as e:
            msg = str(e)
            # wrong key / model / permission: retrying never helps, so stop right here
            if any(c in msg for c in ["Error code: 401", "Error code: 403", "Error code: 404"]):
                raise SystemExit(f"Groq error that retrying will not fix: {msg[:300]}")
            # bad request / too large: skip this batch, it will use the fallback
            if any(c in msg for c in ["Error code: 400", "Error code: 413"]):
                print(f"   batch rejected by Groq ({msg[:120]}), using the fallback for it")
                return ""
            if "per day" in msg.lower():
                print(f"   daily limit reached for {model}, the rest will use the fallback")
                state["stopped"].add(model)
                return ""
            m = re.search(r"try again in\s+(?:(\d+)h)?\s*(?:(\d+)m)?\s*([\d.]+)s", msg)
            wait = (int(m.group(1) or 0) * 3600 + int(m.group(2) or 0) * 60 + float(m.group(3)) + 1) if m else 10 * (attempt + 1)
            if wait > MAX_WAIT:
                print(f"   Groq wants a {wait:.0f}s wait, too long. Stopping calls for {model}, rerun later (cache keeps finished posts)")
                state["stopped"].add(model)
                return ""
            print(f"   API problem ({type(e).__name__}): {msg[:80]} -> retrying in {wait:.0f}s")
            time.sleep(wait)
    return ""


def parse_batch(text, n):
    # Turn the LLM reply into {post number: set of VALID (level1, level2) pairs}. Anything not in the taxonomy is dropped.
    try:
        data = json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text or "", re.S)
        try:
            data = json.loads(m.group(0)) if m else {}
        except Exception:
            data = {}
    if isinstance(data, dict) and isinstance(data.get("labels"), dict):
        data = data["labels"]
    out = {}
    if not isinstance(data, dict):
        return out
    for i in range(1, n + 1):
        items = data.get(str(i), data.get(f"Post {i}", []))
        if isinstance(items, (dict, str)):
            items = [items]
        pairs = set()
        for it in (items if isinstance(items, list) else []):
            if isinstance(it, dict):
                a, b = it.get("level1", ""), it.get("level2", "")
            elif isinstance(it, str) and ">" in it:
                a, b = it.split(">", 1)
            else:
                continue
            key = (str(a).strip().lower(), str(b).strip().lower())
            if key in canon:
                pairs.add(canon[key])
        out[i] = pairs
    return out

In [13]:
# cache so I never pay twice for the same call (only real LLM answers are saved, never failed ones)
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def predict_posts(posts, retriever, model, tag):
    preds, from_llm, todo = {}, set(), []
    for post in posts:
        key = f"{model}||{tag}||{post}"
        if key in cache:
            preds[post] = {tuple(x) for x in cache[key]}
            from_llm.add(post)
        else:
            todo.append(post)
    print(f"{model}: {len(posts) - len(todo)} posts already in cache, {len(todo)} to send to the LLM")

    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo[start:start + BATCH_SIZE]
        reply = ask_llm(model, SYSTEM_PROMPT, build_batch_prompt(batch, retriever, K_EXAMPLES))
        parsed = parse_batch(reply, len(batch)) if reply else {}
        for i, post in enumerate(batch, 1):
            pairs = parsed.get(i, set())
            if pairs:
                preds[post] = pairs
                from_llm.add(post)
                cache[f"{model}||{tag}||{post}"] = sorted(pairs)
            else:
                # LLM gave nothing usable -> fall back to the labels of the most similar training post
                preds[post] = set(retriever.gold[retriever.nearest(post, 1)[0]])
        json.dump(cache, open(CACHE_FILE, "w"))
        print(f"   {model}: {min(start + BATCH_SIZE, len(todo))}/{len(todo)} done")
        if reply:
            time.sleep(state["pause"])

    fallbacks = len(posts) - len(from_llm)
    print(f"{model}: finished {len(posts)} posts | answered by LLM: {len(from_llm)} | fallback: {fallbacks}")
    if fallbacks:
        print("   some posts used the fallback, run this cell again to retry only those")
    return preds

## 6. How I measure accuracy
Every post can have several labels, so plain accuracy is not meaningful. I use set-based **precision, recall and F1** (micro-averaged over all labels), at two levels:
- **Level 1 only**: did it pick the right broad topics?
- **Level 1 + Level 2 pair**: did it pick the exact right factor? (stricter, this is the main number)

I also report **exact match**, the share of posts where the predicted label set is exactly the gold label set.

In [14]:
def score(pred, gold_d, posts, level="pair"):
    tp = fp = fn = exact = 0
    for p in posts:
        g, q = gold_d[p], pred.get(p, set())
        if level == "l1":
            g, q = {a for a, _ in g}, {a for a, _ in q}
        tp += len(g & q); fp += len(q - g); fn += len(g - q)
        exact += int(g == q)
    prec = tp / (tp + fp) if tp + fp else 0
    rec = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0
    return {"precision": round(prec, 3), "recall": round(rec, 3), "f1": round(f1, 3), "exact_match": round(exact / len(posts), 3)}

def report(name, pred):
    r2, r1 = score(pred, gold, val_posts, "pair"), score(pred, gold, val_posts, "l1")
    return {"model": name,
            "L1 P": r1["precision"], "L1 R": r1["recall"], "L1 F1": r1["f1"],
            "Pair P": r2["precision"], "Pair R": r2["recall"], "Pair F1": r2["f1"],
            "Pair exact match": r2["exact_match"]}

## 7. Baseline: TF-IDF + Logistic Regression (no LLM)
Not part of the final solution. I want a classical reference point so I can show how much the LLM really adds.

In [15]:
mlb = MultiLabelBinarizer()
Y_fit = mlb.fit_transform([sorted(gold[p]) for p in fit_posts])   # each label is a (l1, l2) tuple

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), stop_words="english", sublinear_tf=True)),
    ("clf", OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight="balanced"))),
])
baseline.fit(fit_posts, Y_fit)

proba = baseline.predict_proba(val_posts)
baseline_pred = {}
for post, probs in zip(val_posts, proba):
    idx = [i for i, p in enumerate(probs) if p >= 0.35] or [int(probs.argmax())]
    baseline_pred[post] = {mlb.classes_[i] for i in idx}

results = [report("TF-IDF + LogReg (baseline)", baseline_pred)]
pd.DataFrame(results)

,model,L1 P,L1 R,L1 F1,Pair P,Pair R,Pair F1,Pair exact match
0,TF-IDF + LogReg (baseline),0.423,0.684,0.523,0.312,0.608,0.412,0.025


## 8. LLM few-shot on the validation set
The model sees the retrieved examples taken only from the fit split, so the validation is honest. Posts are sent in batches of 10, so this is only a handful of API calls and finishes in a minute or two. If `COMPARE_MODELS = True`, both models get exactly the same prompts.

In [16]:
val_preds = {}
for model in MODELS_TO_RUN:
    val_preds[model] = predict_posts(val_posts, retr_fit, model, tag="val")
    results.append(report(f"{model} (few-shot, k={K_EXAMPLES})", val_preds[model]))

results_df = pd.DataFrame(results).set_index("model")
results_df

openai/gpt-oss-20b: 0 posts already in cache, 40 to send to the LLM
   batch rejected by Groq (Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for), using the fallback for it
   openai/gpt-oss-20b: 10/40 done
   batch rejected by Groq (Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for), using the fallback for it
   openai/gpt-oss-20b: 20/40 done
   batch rejected by Groq (Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for), using the fallback for it
   openai/gpt-oss-20b: 30/40 done
   batch rejected by Groq (Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for), using the fallback for it
   openai/gpt-oss-20b: 40/40 done
openai/gpt-oss-20b: finished 40 posts | answered by LLM: 0 | fallback: 40
   some posts used

,L1 P,L1 R,L1 F1,Pair P,Pair R,Pair F1,Pair exact match
model,,,,,,,
TF-IDF + LogReg (baseline),0.423,0.684,0.523,0.312,0.608,0.412,0.025
"openai/gpt-oss-20b (few-shot, k=3)",0.433,0.382,0.406,0.286,0.278,0.282,0.100


In [17]:
# Which model do I go with? Highest pair-level F1 among the LLMs.
llm_rows = results_df.drop(index="TF-IDF + LogReg (baseline)")
best_name = llm_rows["Pair F1"].idxmax()
BEST_MODEL = best_name.split(" ")[0]
print("Selected model:", BEST_MODEL)
print(llm_rows.loc[best_name])

Selected model: openai/gpt-oss-20b
L1 P                0.433
L1 R                0.382
L1 F1               0.406
Pair P              0.286
Pair R              0.278
Pair F1             0.282
Pair exact match    0.100
Name: openai/gpt-oss-20b (few-shot, k=3), dtype: float64


In [18]:
# Where does the chosen model do well / badly? Per Level 1 factor (pair level)
rows = []
for l1 in sorted(taxonomy):
    tp = fp = fn = 0
    for p in val_posts:
        g = {x for x in gold[p] if x[0] == l1}
        q = {x for x in val_preds[BEST_MODEL][p] if x[0] == l1}
        tp += len(g & q); fp += len(q - g); fn += len(g - q)
    if tp + fn + fp:
        pr = tp / (tp + fp) if tp + fp else 0
        rc = tp / (tp + fn) if tp + fn else 0
        rows.append({"Level 1": l1, "support": tp + fn, "precision": round(pr, 2), "recall": round(rc, 2),
                     "f1": round(2 * pr * rc / (pr + rc), 2) if pr + rc else 0})
per_l1 = pd.DataFrame(rows).sort_values("support", ascending=False)
per_l1

,Level 1,support,precision,recall,f1
13,Tone,13,0.38,0.38,0.38
3,Efficacy,12,0.44,0.33,0.38
11,Sensory,10,0.33,0.10,0.15
8,Pricing,9,0.25,0.22,0.24
7,Moisturization,8,0.67,0.50,0.57
5,Finish,6,0.50,0.33,0.40
10,Safety,4,0.09,0.25,0.13
2,Cleansing,4,0.50,0.50,0.50
4,Feel,4,0.12,0.25,0.17
1,Anti-Aging,3,0.00,0.00,0.00


In [19]:
# A few mistakes to eyeball, useful for the discussion
shown = 0
for p in val_posts:
    g, q = gold[p], val_preds[BEST_MODEL][p]
    if g != q and shown < 8:
        print("POST     :", p)
        print("  gold   :", sorted(g))
        print("  model  :", sorted(q), "\n")
        shown += 1

POST     : I see my skin firmer and more healthy, the relaxing fragrance is a plus especially for a nighttime routine.
  gold   : [('Anti-Aging', 'Elasticity'), ('Sensory', 'Smell'), ('Tone', 'Glow')]
  model  : [('Feel', 'Cooling Sensation'), ('Feel', 'Soothening')] 

POST     : the face wash is good it clears all the dirt and gives the cooling feel
  gold   : [('Cleansing', 'Dirt Removal'), ('Feel', 'Cooling Sensation')]
  model  : [('Cleansing', 'Dirt Removal'), ('Cleansing', 'Oil Removal'), ('Finish', 'Clean'), ('Tone', 'Glow')] 

POST     : Its pure ayurvedic cleanser _ extracts dirts easily. Only three drops is enough for me. Skin doesn't dry.
  gold   : [('Cleansing', 'Dirt Removal'), ('Pricing', 'Value for money'), ('Texture', 'Dryness Control')]
  model  : [('Pricing', 'Affordability'), ('Safety', 'Mildness'), ('Safety', 'Side Effects')] 

POST     : (Basically what the title says, which brand/product you hate most and why??)I personally disagree. FE makes incredible products 

## 9. Final run on `Process.xlsx`
Now I use **all** labelled training posts as the example pool (more examples = better), and label the 190 posts with the selected model.

In [20]:
final_preds = predict_posts(process_posts, retr_full, BEST_MODEL, tag="final")

rows = []
for post in process_posts:
    for l1, l2 in sorted(final_preds[post]):
        rows.append({"Post": post, "Level 1": l1, "Level 2": l2})

output = pd.DataFrame(rows, columns=["Post", "Level 1", "Level 2"])
output.to_excel(OUTPUT_FILE, index=False)
output.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {OUTPUT_FILE} and {OUTPUT_CSV}: {len(output)} rows for {output['Post'].nunique()} posts")
output.head(12)

openai/gpt-oss-20b: 0 posts already in cache, 190 to send to the LLM
   openai/gpt-oss-20b: 10/190 done
   batch rejected by Groq (Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for), using the fallback for it
   openai/gpt-oss-20b: 20/190 done
   batch rejected by Groq (Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for), using the fallback for it
   openai/gpt-oss-20b: 30/190 done
   batch rejected by Groq (Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for), using the fallback for it
   openai/gpt-oss-20b: 40/190 done
   batch rejected by Groq (Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for), using the fallback for it
   openai/gpt-oss-20b: 50/190 done
   batch rejected by Groq (Error code: 400 - {'error

,Post,Level 1,Level 2
0,the fine lines or wrinkles or dark circles or puffiness are as is.,Anti-Aging,Fine Lines
1,"The first time I squeezed a small amount on my palm, I felt it was very creamy.",Sensory,Touch
2,"The first time I used this cream, my skin felt really nourished.",Moisturization,Nourishment
3,The formula changes from cream to foam after putting water. It's worth the money. I wi...,Pricing,Value for money
4,The formula is light and doesn't clog pores.,Acne,Minimize Pores
5,The formula is light and doesn't clog pores.,Finish,Light Weight
6,The formula is water based which makes your skin super glowy and even provide hydratio .,Moisturization,Hydration
7,The formula is water based which makes your skin super glowy and even provide hydratio .,Tone,Glow
8,The fragrance is so amazing and my face glows after every wasg,Sensory,Smell
9,The fragrance is so amazing and my face glows after every wasg,Tone,Glow


In [21]:
# Sanity checks before I submit
assert set(output["Post"]) == set(process_posts), "some posts are missing from the output"
assert all((a, b) in valid_pairs for a, b in zip(output["Level 1"], output["Level 2"])), "invalid pair found"
assert not output.duplicated().any(), "duplicate rows found"
print("All", len(process_posts), "posts labelled, all pairs valid, no duplicates.")
print("Avg labels per post (output):", round(len(output) / output["Post"].nunique(), 2),
      "| in training:", round(len(train) / train["Post"].nunique(), 2))
print()
print(output["Level 1"].value_counts())

All 190 posts labelled, all pairs valid, no duplicates.
Avg labels per post (output): 1.89 | in training: 1.92

Level 1
Sensory           56
Tone              49
Efficacy          48
Moisturization    43
Pricing           30
Texture           29
Finish            28
Safety            25
Feel              20
Acne              10
Anti-Aging         6
Cleansing          6
Lasting            5
Relief             3
Health             1
Practicality       1
Name: count, dtype: int64


## 10. Write-up (approach, models considered, accuracy)

In [22]:
b = results_df.loc[best_name]
base = results_df.loc["TF-IDF + LogReg (baseline)"]
model_lines = "\n".join(f"- {name} (few-shot): pair F1 {row['Pair F1']}" for name, row in llm_rows.iterrows())

summary = f"""
APPROACH
- Parsed Training.xlsx into a taxonomy of {len(taxonomy)} Level 1 factors and {len(valid_pairs)} valid Level 1/Level 2 pairs.
- For every post, retrieved the {K_EXAMPLES} most similar labelled posts (TF-IDF cosine similarity) and passed them as
  few-shot examples, together with the complete taxonomy, to a Groq-hosted LLM (temperature 0, JSON output, {BATCH_SIZE} posts per request).
- Parsed the JSON and dropped any pair that is not in the taxonomy, so no invented labels reach the output.
- Posts can have multiple labels; the output has one row per (Post, Level 1, Level 2) mapping, same format as Training.xlsx.

MODELS CONSIDERED
- BERT-style models: not allowed by the exercise.
- TF-IDF + Logistic Regression (classical baseline, not an LLM): pair F1 {base['Pair F1']}. Trained on only ~{len(fit_posts)} posts,
  so it has very little data for the rare factors.
{model_lines}
- Selected: {BEST_MODEL}. It is free on Groq, needs no training, and handles rare factors because the taxonomy is in the prompt.

OUTPUT ACCURACY (validation: {len(val_posts)} held-out training posts, examples retrieved only from the other {len(fit_posts)})
- Level 1 only : precision {b['L1 P']}, recall {b['L1 R']}, F1 {b['L1 F1']}
- Level 1 + 2  : precision {b['Pair P']}, recall {b['Pair R']}, F1 {b['Pair F1']}
- Exact match of the full label set per post: {b['Pair exact match']}
- Process.xlsx has no ground truth, so these validation numbers are the estimate of the expected accuracy.
  The final run uses all {len(all_posts)} labelled posts as examples, so real accuracy should be similar or slightly better.

LIMITATIONS / NEXT STEPS
- The validation set is small, so scores are a bit noisy.
- Gold labels look subjective (same sentence can reasonably get different labels); part of the error is label noise.
- Possible improvements: tune K, add short definitions per factor, run a self-consistency vote over several samples,
  or fine-tune a small open model on the labelled data.
"""
print(summary)
open("approach_summary.txt", "w").write(summary)


APPROACH
- Parsed Training.xlsx into a taxonomy of 17 Level 1 factors and 43 valid Level 1/Level 2 pairs.
- For every post, retrieved the 3 most similar labelled posts (TF-IDF cosine similarity) and passed them as
  few-shot examples, together with the complete taxonomy, to a Groq-hosted LLM (temperature 0, JSON output, 10 posts per request).
- Parsed the JSON and dropped any pair that is not in the taxonomy, so no invented labels reach the output.
- Posts can have multiple labels; the output has one row per (Post, Level 1, Level 2) mapping, same format as Training.xlsx.

MODELS CONSIDERED
- BERT-style models: not allowed by the exercise.
- TF-IDF + Logistic Regression (classical baseline, not an LLM): pair F1 0.412. Trained on only ~200 posts,
  so it has very little data for the rare factors.
- openai/gpt-oss-20b (few-shot, k=3) (few-shot): pair F1 0.282
- Selected: openai/gpt-oss-20b. It is free on Groq, needs no training, and handles rare factors because the taxonomy is in the pro

1864